# Figures for docs/approach.md

Runs top to bottom with `jupyter nbconvert --execute docs/figures.ipynb`. PNGs go to `docs/fig/`.

In [1]:
import math, os, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from sim.arena import Arena
from sim.game import Game
from sim.cards import create

# nbconvert runs with cwd = docs/; a plain kernel from the repo root does not
FIG = 'fig' if os.path.basename(os.getcwd()) == 'docs' else 'docs/fig'
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 9, 'figure.dpi': 150, 'savefig.dpi': 200})
BLUE, RED = '#2166ac', '#b2182b'
A = Arena()

def draw_arena(ax):
    for y in range(A.H):
        for x in range(A.W):
            c = A.grid[y][x]
            if c == 'R': col = '#9ecae1'
            elif c == 'B': col = '#c49a6c'
            elif c: col = '#f4a582' if c[0] == 'R' else '#92c5de'
            else: continue
            ax.add_patch(Rectangle((x, y), 1, 1, fc=col, ec='none'))
    ax.set_xticks(range(0, 19), minor=True); ax.set_yticks(range(0, 33), minor=True)
    ax.grid(which='minor', color='0.85', lw=0.3)
    ax.set_xlim(0, 18); ax.set_ylim(0, 32); ax.set_aspect('equal')
    ax.set_xlabel('x (tiles)'); ax.set_ylabel('y (tiles)')

## Figure 1: arena geometry, interaction radius R, half-tile raster

In [2]:
R = 10.0
fig, ax = plt.subplots(figsize=(4.2, 7))
draw_arena(ax)
ax.add_patch(Rectangle((0, 0), 18, 15, fc='none', ec=BLUE, ls='--', lw=0.8))
ax.add_patch(Rectangle((0, 17), 18, 15, fc='none', ec=RED, ls='--', lw=0.8))
cx, cy = 4.5, 16.0
ax.add_patch(Circle((cx, cy), R, fc='#fdd49e', ec='#e6550d', alpha=0.35, lw=1.2))
ax.plot(cx, cy, 'o', color='#e6550d', ms=4)
ax.text(cx + 0.4, cy + 0.4, 'unit on left bridge', fontsize=7, color='#e6550d')
ax.text(9.6, 25.3, 'R = 10 tiles', fontsize=8, color='#e6550d')
x0, y0, w, h = 11, 8, 6, 6
for i in range(2 * w + 1): ax.plot([x0 + i / 2] * 2, [y0, y0 + h], color='k', lw=0.4)
for j in range(2 * h + 1): ax.plot([x0, x0 + w], [y0 + j / 2] * 2, color='k', lw=0.4)
ax.text(x0 + w / 2, y0 + h + 0.4, 'half-tile cells', ha='center', fontsize=8)
ax.text(9, 0.4, 'blue deploy zone', ha='center', fontsize=7, color=BLUE)
ax.text(9, 31.0, 'red deploy zone', ha='center', fontsize=7, color=RED)
fig.savefig(f'{FIG}/arena_radius.png', bbox_inches='tight')
plt.close(fig)

## Figure 2: receptive field coverage of a neighborhood-attention stack

In [3]:
Rc = 20  # R = 10 tiles in half-tile cells
Ls = np.arange(1, 21)
ws = (3, 5, 7, 9, 11)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.1))
mins = []
for w in ws:
    r = Ls * (w - 1) // 2
    a1.plot(Ls, r, marker='o', ms=3, label=f'w = {w}')
    Lmin = math.ceil(Rc / ((w - 1) / 2))
    mins.append((w, Lmin))
    a1.plot(Lmin, Lmin * (w - 1) // 2, 'k*', ms=9, zorder=5)
a1.axhline(Rc, color='k', ls='--', lw=0.8)
a1.text(1, Rc + 1.2, 'R = 10 tiles = 20 cells', fontsize=8)
a1.set_xlabel('layers L'); a1.set_ylabel('receptive radius L(w-1)/2 (cells)')
a1.set_ylim(0, 45); a1.legend(fontsize=7, ncol=2, loc='upper left')
a1.set_title('stars: minimal L that covers R', fontsize=8)
a2.bar([str(w) for w, _ in mins], [w * w * L for w, L in mins], color='0.5')
for i, (w, L) in enumerate(mins):
    a2.text(i, w * w * L + 8, f'L = {L}', ha='center', fontsize=7)
a2.set_xlabel('window w (cells)'); a2.set_ylabel('attention pairs per cell, w^2 L')
a2.set_title('cost of the minimal configuration', fontsize=8)
fig.tight_layout()
fig.savefig(f'{FIG}/receptive_field.png', bbox_inches='tight')
plt.close(fig)

## Figure 3: compounding factor of the autoregressive error bound

In [4]:
H = np.linspace(0.25, 60, 400)
ks = (1, 4, 16, 64)
Lvals = (1.0, 1.01, 1.05)
fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=True)
for ax, L in zip(axes, Lvals):
    for k in ks:
        T = 20 * H / k
        f = T if L == 1.0 else (L**T - 1) / (L - 1)
        ax.semilogy(H, f, label=f'k = {k} ticks')
    for s in (10, 30): ax.axvline(s, color='0.8', lw=0.6)
    ax.set_title(f'L_F = {L}'); ax.set_xlabel('horizon (s)')
axes[0].set_ylabel('(L_F^T - 1)/(L_F - 1),  T = 20 H / k')
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(f'{FIG}/compounding.png', bbox_inches='tight')
plt.close(fig)

## Figure 4: supervision density

In [5]:
ticks = 3600  # 3 min regulation at 20 Hz
cells_ = 36 * 64
rows = [('decision labels\n(human replay)', 40),
        ('global-block labels\n(one per tick)', ticks),
        ('entity labels\n(about 20 per tick)', ticks * 20),
        ('cell labels\n(2304 per tick)', ticks * cells_)]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.5, 3.1))
a1.barh([r[0] for r in rows], [r[1] for r in rows], color=['0.6', '0.6', '0.6', '#e6550d'])
a1.set_xscale('log'); a1.set_xlabel('supervised examples per game')
for i, (_, v) in enumerate(rows): a1.text(v * 1.3, i, f'{v:,.0f}', va='center', fontsize=7)
a1.set_xlim(10, 3e8)
n = np.logspace(0, 6, 200)
a2.loglog(n, 40 * n, color='0.4', label='decision labels, human games')
a2.loglog(n, ticks * cells_ * n, color='#e6550d', label='cell labels, simulated games')
a2.axvline(168e3, color=BLUE, ls='--', lw=0.8)
a2.text(168e3 * 1.15, 3e2, '168K clean 1v1\nreplays', fontsize=7, color=BLUE)
a2.set_xlabel('games'); a2.set_ylabel('examples'); a2.legend(fontsize=7, loc='upper left')
fig.tight_layout()
fig.savefig(f'{FIG}/supervision_density.png', bbox_inches='tight')
plt.close(fig)

## Figure 5: a simulator state and its half-tile raster

In [6]:
random.seed(0)
g = Game()
def put(tm, card, x, y):
    r = create(card, 11, tm, float(x), float(y))
    for u in (r if isinstance(r, list) else [r]): g.deploy(tm, u)
g.run_to(50); put('blue', 'giant', 3, 13); put('blue', 'musketeer', 4, 11)
g.run_to(56); put('red', 'skeleton_army', 3, 19); put('red', 'knight', 5, 20)
put('blue', 'hog_rider', 14, 15); put('red', 'archers', 14, 21)
g.run_to(60)
units = [(u.team, u.x, u.y, u.hp / u.max_hp, u.name) for tm in ('blue', 'red') for u in g.players[tm].troops]
cnt = np.zeros((64, 36))
for _, x, y, _, _ in units:
    cnt[min(63, int(2 * y)), min(35, int(2 * x))] += 1

fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 6.4))
draw_arena(a1)
for t in g.arena.towers:
    a1.text(t.cx, t.cy, f'{t.hp}', ha='center', va='center', fontsize=6)
for tm, x, y, hp, _ in units:
    a1.plot(x, y, 'o', ms=2.5 + 4 * hp, color=BLUE if tm == 'blue' else RED, alpha=0.75, mec='none')
a1.set_title(f'{len(units)} entities at t = {g.t:.0f} s (marker size = hp fraction)', fontsize=8)
cmap = plt.get_cmap('magma_r').copy(); cmap.set_bad('white')
im = a2.imshow(np.ma.masked_equal(cnt, 0), origin='lower', cmap=cmap, extent=(0, 18, 0, 32), interpolation='nearest', vmin=0)
a2.set_title(f'entity count per half-tile cell (max {int(cnt.max())})', fontsize=8)
a2.set_xlabel('x (tiles)'); a2.set_yticks([]); a2.set_xticks(range(0, 19, 5))
for sp in a2.spines.values(): sp.set_color('0.7')
fig.colorbar(im, ax=a2, fraction=0.04, pad=0.02)
fig.tight_layout()
fig.savefig(f'{FIG}/sim_raster.png', bbox_inches='tight')
plt.close(fig)
print(g.status())
print(sorted(set(n for *_, n in units)), 'occupied cells:', int((cnt > 0).sum()))

T=60.0s Phase=regulation
  blue: 0cr 10.0ex
  red: 0cr 10.0ex
  blue king: 4824/4824  
  blue princess: 3052/3052 *
  blue princess: 3052/3052 *
  red king: 4824/4824  
  red princess: 3052/3052 *
  red princess: 2680/3052 *

['Archers', 'Hog Rider', 'Knight', 'Musketeer', 'Skeleton Army'] occupied cells: 8
